# Discovering Agents with MCP

<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/agent_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Most agent examples start from a tool or service you already know about. This one starts from a task and *no* candidates: the agent queries a public registry to find out, at runtime, which third-party agents can do the job.

The registry used here is [Aidress](https://api.aidress.ai), which exposes its lookups over MCP. Because the agents that discovery turns up are ones you have never worked with, the registry also returns trust data for each candidate — a trust score, whether the agent is verified, how many transactions it has completed, and any flags — so the agent can vet a candidate before recommending it.

In [ ]:
%pip install llama-index-tools-mcp llama-index-llms-openai

## Connecting to the registry

The registry exposes read-only lookups alongside tools that mutate registry state, such as registering an agent or proxying a paid call. Discovery needs only the read-only subset, so `allowed_tools` keeps the agent from reaching the rest.

The read-only lookups are public and need no credentials.

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, aget_tools_from_mcp_url

REGISTRY_URL = "https://api.aidress.ai/mcp/sse"

READ_ONLY_TOOLS = [
    # Discovery: find candidates by capability, or page through the whole registry.
    "match_agents",
    "list_registry",
    # Due diligence on a candidate that discovery surfaced.
    "verify_agent",
    "get_agent",
    "protocol_reference",
]

client = BasicMCPClient(REGISTRY_URL)
tools = await aget_tools_from_mcp_url(
    REGISTRY_URL, client=client, allowed_tools=READ_ONLY_TOOLS
)

print(f"{len(tools)} tools available:")
for tool in tools:
    print(f"  - {tool.metadata.name}")

## Discovering candidates for a task

Now give an agent the tools and a task, without naming a single counterparty. It has to find out who exists.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

agent = FunctionAgent(
    name="Agent Discovery Assistant",
    description="Finds third-party agents that can do a task, and vets them.",
    llm=OpenAI(model="gpt-4o"),
    tools=tools,
    system_prompt=(
        "You find third-party agents that can do a task the user needs done. "
        "Start by searching the registry for agents offering the required "
        "capability, and report what you found: how many candidates there are "
        "and what each one does. Then, because the user has not worked with any "
        "of them before, check each candidate's trust score, whether it is "
        "verified, how many transactions it has completed, and any flags. Close "
        "with the candidate you would pick and the evidence behind the choice. "
        "Report only the values the tools return; never invent an agent or a score."
    ),
)

response = await agent.run(
    "I need a web research task done, but I don't know which agents offer that. "
    "Find the ones that do, then tell me which of them I can trust with the job."
)
print(response)

The registry holds real third-party agents, so the candidates and their numbers change over time — nothing above is hard-coded, and rerunning this may surface a different set.